# Extracting and attributing quotations to named scholars

This notebook documents the workflow used to identify quotations in Sathas’s text, as a samlple attribute them to named speakers or sources, manually inspect and correct the resulting attributions, and reorganize the quotations by scholar.


In [2]:
import pandas as pd
import re
import string
from tqdm import tqdm

In [3]:
# import preprocessed history
df1 = pd.read_csv("Sathas_Preprocessed.csv")

In [5]:
df1['Chapters'].unique()

<ArrowStringArray>
['Preface', 'Chapter_1', 'Chapter_2', 'Chapter_3', 'Chapter_4', 'Chapter_5']
Length: 6, dtype: str

In [9]:
# load necessary columns
df_quotes = df1[['Chapters', 'text_monotonic', 'text_no_quotes_monotonic]]

In [10]:
df_quotes.head(1)

,Chapters,text_monotonic_accents
0,Preface,. Μέρος της υπό του Εθνικού Πανεπιστημίου βραβ...


In [11]:
#remove final random punctuation
def clean_flying_punctuation(text):
    # Remove specified "flying" punctuation marks
    cleaned_text = re.sub(r"[΄῾˙῎\"]", "", text)
    
    # Normalize redundant whitespaces (replace multiple spaces with a single space)
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()
    
    return cleaned_text

# Apply the function to the entire column
df_quotes['text_monotonic_accents'] = df_quotes['text_monotonic_accents'].apply(clean_flying_punctuation)


**Processing strategy.** The analysis was carried out chapter by chapter. To keep the notebook readable, the cells below demonstrate the full extraction and attribution procedure using Chapter 5 (`test5`) as an illustrative example. The same workflow was applied separately to the remaining chapters; their processed results are concatenated later in the notebook.


In [12]:
# separate in chapters
test0 = df_quotes[0:1]
test1 = df_quotes[1:2]
test2 = df_quotes[2:3]
test3 = df_quotes[3:4]
test4 = df_quotes[4:5]
test5 = df_quotes[5:]


In [ ]:
# Check and print the filtered text
filtered_text = test5['text_monotonic']
print(filtered_text)


## 1. Extract quotations and their positions


In [54]:
def extract_quotes_with_positions(text):
    results = {}

    if not isinstance(text, str):
        return results

    matches = list(re.finditer(r'«(.*?)»', text, flags=re.DOTALL))

    for i, m in enumerate(matches, start=1):
        results[f"Quote_{i}"] = m.group(1).strip()
        results[f"Quote_{i}_start"] = m.start()

    return results

In [55]:
quote_dicts = test5["text_monotonic_accents"].apply(extract_quotes_with_positions)

quote_df = pd.DataFrame(
    quote_dicts.tolist(),
    index=test5.index
)

test5 = pd.concat([test5, quote_df], axis=1)

In [56]:
#display the texts better
pd.set_option('display.width', 50)        
pd.set_option('display.max_colwidth', 50)

In [ ]:
#check
test5

In [58]:
test5.columns

Index(['Chapters', 'text_monotonic_accents',
       'Quote_1', 'Quote_1_start', 'Quote_2',
       'Quote_2_start', 'Quote_3',
       'Quote_3_start', 'Quote_4',
       'Quote_4_start', 'Quote_5',
       'Quote_5_start', 'Quote_6',
       'Quote_6_start', 'Quote_7',
       'Quote_7_start', 'Quote_8',
       'Quote_8_start', 'Quote_9',
       'Quote_9_start', 'Quote_10',
       'Quote_10_start', 'Quote_11',
       'Quote_11_start', 'Quote_12',
       'Quote_12_start', 'Quote_13',
       'Quote_13_start', 'Quote_14',
       'Quote_14_start', 'Quote_15',
       'Quote_15_start', 'Quote_16',
       'Quote_16_start', 'Quote_17',
       'Quote_17_start', 'Quote_18',
       'Quote_18_start', 'Quote_19',
       'Quote_19_start', 'Quote_20',
       'Quote_20_start', 'Quote_21',
       'Quote_21_start', 'Quote_22',
       'Quote_22_start', 'Quote_23',
       'Quote_23_start', 'Quote_24',
       'Quote_24_start', 'Quote_25',
       'Quote_25_start', 'Quote_26',
       'Quote_26_start', 'Quote_27',

## 2. Attribute quotations to scholars with an LLM

The model receives each quotation together with the local textual context immediately preceding it and returns the explicitly associated speaker or source.


In [23]:
from tqdm import tqdm  # Progress bar
import json

In [24]:
import openai
from openai import OpenAI
client = OpenAI()

In [59]:
def identify_speaker_from_position(text, quote, start_pos, context_chars=1200):
    """
    Identify the speaker/source of a quotation using only
    the local context immediately preceding the quotation.
    """

    if pd.isna(quote) or str(quote).strip() == "":
        return "Unattributed"

    if pd.isna(text) or pd.isna(start_pos):
        return "Unattributed"

    start_pos = int(start_pos)

    # Context immediately before the opening «
    before = text[max(0, start_pos - context_chars):start_pos]

    prompt = f"""
You are analyzing a nineteenth-century Greek historical text.

Your task is to identify the person whose words are introduced
by the quotation.

QUOTATION:
«{quote}»

TEXT IMMEDIATELY BEFORE THE QUOTATION:
{before}

Instructions:
1. Identify the author, speaker, or source explicitly associated
   with the quotation.
2. Focus especially on the final sentences immediately before
   the quotation.
3. Look for attribution formulas such as:
   "γράφει", "έγραφεν", "λέγει", "είπεν", "αναφέρει",
   "παρατηρεί", "παρατίθημι", "παρατιθέμεθα",
   "κατά τον", "ιδού ... έγραφεν".
4. Do NOT assume that the narrator of the surrounding text is
   the speaker.
5. Ignore unrelated names mentioned earlier in the context.
6. If the attribution is genuinely impossible, return:
   Unattributed

Return ONLY the person's name.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=50,
            temperature=0
        )

        speaker = response.choices[0].message.content.strip()

        return speaker if speaker else "Unattributed"

    except Exception as e:
        print(f"Error: {e}")
        return "Unattributed"

In [60]:
from tqdm.auto import tqdm

quote_numbers = range(1, 31)

pbar = tqdm(quote_numbers, desc="Identifying speakers")

for i in pbar:
    quote_col = f"Quote_{i}"
    start_col = f"Quote_{i}_start"
    speaker_col = f"Quote_{i}_speaker"

    test5[speaker_col] = test5.apply(
        lambda row: identify_speaker_from_position(
            row["text_monotonic_accents"],
            row[quote_col],
            row[start_col]
        ),
        axis=1
    )

    speaker = test5.iloc[0][speaker_col]

    tqdm.write(
        f"Quote_{i}: {speaker}"
    )

Identifying speakers:   0%|          | 0/30 [00:00<?, ?it/s]

Quote_1: Παναγιώτης Κοδρικάς
Quote_2: Κοδρικάς
Quote_3: Κοραής
Quote_4: Κοραής
Quote_5: Κοραής
Quote_6: Κοραής
Quote_7: Δημήτριος ο Δάρβαρις
Quote_8: Γεώργιος Κρομμύδας
Quote_9: Παύλου
Quote_10: Στέφανος Κομμητάς
Quote_11: Κομμητάς
Quote_12: Νεόφυτος ο Δούκας
Quote_13: Δούκας
Quote_14: Unattributed
Quote_15: Ιγνάτιος Σκαλιώρας
Quote_16: ᾿Ανδρέσο Θεοτόκη
Quote_17: Unattributed
Quote_18: Κοδρικάς
Quote_19: Κοδρικάς
Quote_20: Κοραής
Quote_21: Πατρινός
Quote_22: Νεόφυτος Δούκας
Quote_23: Μαρκέζος Βασίλειος Πουότης
Quote_24: Νεόφυτος ο Δούκας
Quote_25: Δούκας
Quote_26: Δούκας
Quote_27: Γρηγόριος Ε
Quote_28: Κωνσταντίνος Ηλιάδης
Quote_29: Ιακωβάκης Νερουλός
Quote_30: Κωνσταντίνου Οικονόμου


In [ ]:
# print for checking
for i in range(1, 31):
    print(
        f"Quote_{i}: "
        f"{test5.iloc[0][f'Quote_{i}_speaker']}"
    )

### Inspect the model attributions
Example with chapter 5


In [62]:
test5

,Chapters,text_monotonic_accents,Quote_1,Quote_1_start,Quote_2,Quote_2_start,Quote_3,Quote_3_start,Quote_4,Quote_4_start,Quote_5,Quote_5_start,Quote_6,Quote_6_start,Quote_7,Quote_7_start,Quote_8,Quote_8_start,Quote_9,Quote_9_start,Quote_10,Quote_10_start,Quote_11,Quote_11_start,Quote_12,Quote_12_start,Quote_13,Quote_13_start,Quote_14,Quote_14_start,Quote_15,Quote_15_start,Quote_16,Quote_16_start,Quote_17,Quote_17_start,Quote_18,Quote_18_start,Quote_19,Quote_19_start,Quote_20,Quote_20_start,Quote_21,Quote_21_start,Quote_22,Quote_22_start,Quote_23,Quote_23_start,Quote_24,Quote_24_start,Quote_25,Quote_25_start,Quote_26,Quote_26_start,Quote_27,Quote_27_start,Quote_28,Quote_28_start,Quote_29,Quote_29_start,Quote_30,Quote_30_start,Quote_1_speaker,Quote_2_speaker,Quote_3_speaker,Quote_4_speaker,Quote_5_speaker,Quote_6_speaker,Quote_7_speaker,Quote_8_speaker,Quote_9_speaker,Quote_10_speaker,Quote_11_speaker,Quote_12_speaker,Quote_13_speaker,Quote_14_speaker,Quote_15_speaker,Quote_16_speaker,Quote_17_speaker,Quote_18_speaker,Quote_19_speaker,Quote_20_speaker,Quote_21_speaker,Quote_22_speaker,Quote_23_speaker,Quote_24_speaker,Quote_25_speaker,Quote_26_speaker,Quote_27_speaker,Quote_28_speaker,Quote_29_speaker,Quote_30_speaker
5,Chapter_5,. 1794-1821. Παναγιώτης Κοδρικάς. – ᾿Αδαμάντιο...,"Δυσκολώτερον, και αργότερον πράγμα δεν με φαίν...",2278,"Οτι η αρχαία των Ελλήνων φωνή, άμα της εν πολι...",18249,Κατήχησις Πλάτωνος. Εν Λιψία 1782. Δεν είναι κ...,55086,Το πρώτον έργον του θέλοντος να πείση είναι να...,78963,Εις τούτο λοιπόν κυρίως της Ρητορικής το μέρος...,83643,Πρώτην φοράν σήμερον η Ελλάς μετά την μακροχρό...,106294,Η κοινή των Γραικών διάλεκτος διαφέρει πολύ απ...,127485,"Αναλογιζομένω μοι θάμά, και ταλανίζοντι προς π...",131823,"Πρός τοις πολλοίς άλλοις, οίς κατά του εχθρού ...",133351,"Εάν την αισχίστην αποδυσάμενοι αιδώ, εθίσωμεν ...",134820,"Ο συγγραφεύς της Γραμματικής, η επιγέγραπται, ...",140161,Διαιρούμεν λοιπόν την γλώσσαν των νύν γραφόντω...,157449,"Πρό δεκαετίας, γράφει, οι του γένους ελλόγιμοι...",169769,"Λόγους γράφετε η νόμους, και αποφθέγματα; Ελλη...",171754,Τώ όντι φίλε! πότην εντροπήν δεν θέλει αισθανθ...,173537,Διό ου δεί ημάς Έλληνας όντας Ιταλοτουρκογραικ...,177840,Θεός ηγείσθω παντός έργου και λόγου. Επί καθηγ...,179808,όχι μόνον δεν εφαντάσθησάν ποτε τας εν εκείνω ...,182142,Έμελλεν άρα τώρα στα γηράματα να μάς μάθη και ...,182563,Τιμιώτατε κύριε Πατρινέ. Πρό ολίγων ημερών εξέ...,186567,"Εγώ, σοφολογιώτατε, μήτε φιλόσοφος είμαι, μήτε...",188123,Εάν οι τα πρώτα των καθ᾿ ημάς της σοφίας επαγγ...,190415,"Ανδρες Ελληνες. Δεν αμφιβάλλεται, ότι σκοπός τ...",194530,"Ειρήνην την εμήν, ώ παίδες εμοί, ειρήνην την ε...",202607,Τίς ουχ ορά τα ενδοξότερα των σχολείων τα μεν ...,204881,"Περί δε της φιλολογίας, εγώ μεν περί γλώσσης, ...",205376,Ότι εις κενόν απέβησαν αι περί γλώσσης διδασκα...,206222,Βλέπων την μεγίστην απώλειάν τινων της νεολαία...,207088,"Πράξις Β, σκηνή Γ. Μιτυληνηοί. - ᾿Αφεντέλη, με...",208378,"Σεργίου Ιωάννου, ᾿Αναίρεσις νοσοκομικής αναφορ...",212818,Παναγιώτης Κοδρικάς,Κοδρικάς,Κοραής,Κοραής,Κοραής,Κοραής,Δημήτριος ο Δάρβαρις,Γεώργιος Κρομμύδας,Παύλου,Στέφανος Κομμητάς,Κομμητάς,Νεόφυτος ο Δούκας,Δούκας,Unattributed,Ιγνάτιος Σκαλιώρας,᾿Ανδρέσο Θεοτόκη,Unattributed,Κοδρικάς,Κοδρικάς,Κοραής,Πατρινός,Νεόφυτος Δούκας,Μαρκέζος Βασίλειος Πουότης,Νεόφυτος ο Δούκας,Δούκας,Δούκας,Γρηγόριος Ε,Κωνσταντίνος Ηλιάδης,Ιακωβάκης Νερουλός,Κωνσταντίνου Οικονόμου


## 3. Manual inspection and correction of scholar names

The automatically generated attributions were manually inspected and validated. The corrections below record the reviewed attribution for the -illustrative- chapter.


In [63]:
test5.at[5, 'Quote_2_speaker'] = 'Κοδρικάς'
test5.at[5, 'Quote_7_speaker'] = 'Δημήτριος Δάρβαρις'
test5.at[5, 'Quote_9_speaker'] = 'Αυτοκράτορας Παύλος'
test5.at[5, 'Quote_10_speaker'] = 'Κομμητάς'
test5.at[5, 'Quote_12_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_13_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_14_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_16_speaker'] = 'Ανδρέας Θεοτόκης'
test5.at[5, 'Quote_17_speaker'] = 'Ψήφισμα Σχολής Κυδωνιών'
test5.at[5, 'Quote_18_speaker'] = 'Έλληνες Κοραϊστές Σπουδαστές'
test5.at[5, 'Quote_23_speaker'] = 'Βασίλειος Πουότης'
test5.at[5, 'Quote_24_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_25_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_26_speaker'] = 'Νεόφυτος Δούκας'
test5.at[5, 'Quote_30_speaker'] = 'Κωνσταντίνος Οικονόμου'


## 4. Reorganize quotations by scholar


In [66]:
from collections import defaultdict

def reorganize_quotes_by_speaker(df):
    """
    Create new columns named after the scholars and merge quotes from the same speaker.
    """
    speaker_quotes = defaultdict(list)

    # Iterate through all Quote_X and Quote_X_speaker columns
    for i in range(1, 38):
        quote_col = f"Quote_{i}"
        speaker_col = f"Quote_{i}_speaker"
        
        if quote_col in df.columns and speaker_col in df.columns:
            for idx, speaker in df[speaker_col].items():
                if pd.notna(speaker) and speaker != "Unattributed":
                    # Clean speaker name by adding underscores
                    clean_name = speaker.replace(' ', '_').replace('.', '').replace(',', '')
                    # Append quote to the correct speaker's list
                    speaker_quotes[(idx, clean_name)].append(df.at[idx, quote_col])

    # Add merged quotes as new columns
    for (idx, name), quotes in speaker_quotes.items():
        col_name = f"{name}_quote"
        merged_quote = ' '.join(quotes)
        df.at[idx, col_name] = merged_quote


    return df

In [67]:
# Apply the transformation
test5 = reorganize_quotes_by_speaker(test5)

In [ ]:
test5.columns

In [71]:
# Drop old Quote_X and Quote_X_speaker columns
columns_to_drop = [col for col in test5.columns if col.startswith('Quote_')]
test5.drop(columns=columns_to_drop, inplace=True, errors='ignore')


In [74]:
test5.columns

Index(['Chapters', 'text_monotonic_accents',
       'Παναγιώτης_Κοδρικάς_quote',
       'Κοδρικάς_quote', 'Κοραής_quote',
       'Δημήτριος_Δάρβαρις_quote',
       'Γεώργιος_Κρομμύδας_quote',
       'Αυτοκράτορας_Παύλος_quote',
       'Κομμητάς_quote', 'Νεόφυτος_Δούκας_quote',
       'Ιγνάτιος_Σκαλιώρας_quote',
       'Ανδρέας_Θεοτόκης_quote',
       'Ψήφισμα_Σχολής_Κυδωνιών_quote',
       'Έλληνες_Κοραϊστές_Σπουδαστές_quote',
       'Πατρινός_quote',
       'Βασίλειος_Πουότης_quote',
       'Γρηγόριος_Ε_quote',
       'Κωνσταντίνος_Ηλιάδης_quote',
       'Ιακωβάκης_Νερουλός_quote',
       'Κωνσταντίνος_Οικονόμου_quote'],
      dtype='str')

## 5. Save each processed chapter (optional)


In [378]:
#test0.to_csv('Sathas_Prologos_with_Quotes.csv', index=False)

In [424]:
#test1.to_csv('Sathas_Chap1_with_Quotes.csv', index=False)

In [475]:
#test2.to_csv('Sathas_Chap2_with_Quotes.csv', index=False)

In [505]:
#test3.to_csv('Sathas_Chap3_with_Quotes.csv', index=False)

In [52]:
#test4.to_csv('Sathas_Chap4_with_Quotes.csv', index=False)

In [75]:
test5.to_csv('Sathas_Chap5_with_Quotes.csv', index=False)

## 6. Concatenate chapter-level results


In [471]:
# List all chapter DataFrames
chapter_dfs = [test0, test1, test2, test3, test4, test5]

# Concatenate them vertically (stacked one under the other)
sathas_df_quotes = pd.concat(chapter_dfs, ignore_index=True)


In [ ]:
# check
sathas_df_quotes

### Save the combined quotation data


In [473]:
sathas_df_quotes.to_csv('Sathas_with_quotes.csv', index=False)